In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Task 1: Write your code here:
data_path = os.path.join(path, 'Q1_data.csv')
df_data = pd.read_csv(data_path)

In [ ]:
# Task 2: Write your code here:
print(f"Dataset shape: {df_data.shape}")
df_data.head()

In [ ]:
# Task 3: Write your code here:
df_data.info()

In [ ]:
# Task 4: Write your code here:
df_data.describe()

In [ ]:
# Task 5: Write your code here:
df_data.Delivery_Time.hist()

In [ ]:
# Task 1: Write your code here:
data = df_data.drop('Order_ID', axis=1)

In [ ]:
data

In [ ]:
# Task 2: Write your code here:
# Check missing values
missing_values = data.isnull().sum()
print("Columns with missing values:")
print(missing_values[missing_values > 0])

# Fill categorical columns with 'unknown' for categorical columns
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    data[col] = data[col].fillna('unknown')

# # Fill missing values with mean for numerical columns
data['Courier_Experience_yrs'] = data['Courier_Experience_yrs'].fillna(data['Courier_Experience_yrs'].mean())
data['Delivery_Time'] = data['Delivery_Time'].fillna(data['Delivery_Time'].mean())

print("Missing values remaining:", data.isnull().sum().sum())



In [ ]:
# Task 3: Write your code here:
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(data)
data

In [ ]:
# Task 4: Write your code here:
categorical_cols = data.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))

data.head()


In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

features = data.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
data[features] = scaler.fit_transform(data[features])
data.head()

In [ ]:
# Task 6: Write your code here:
data["Delivery_Time"].hist()

In [ ]:
# Task 1: Write your code here:
X = data.drop("Delivery_Time", axis=1).astype(float)
y = data['Delivery_Time'].astype(float)

In [ ]:
# BCE in NumPy
def binary_cross_entropy(y, y_hat):
  epsilon = 1e-15  # Very small number to prevent log(0)
  y_hat = np.clip(y_hat, epsilon, 1 - epsilon) # np.clip(value, min, max)

  loss = -1/len(y) * np.sum(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))
  return loss

In [ ]:
from tqdm import tqdm
def gradient_descent(X, y, learning_rate, n_iters=500):
  m, n = X.shape # m rows, n columns (dimensions)
  theta = np.zeros(n) # initialize a zeros weight vector with n dimensions
  losses = []

  for _ in tqdm(range(n_iters), desc="Training Logistic Regression"):
    z = np.dot(X, theta)
    y_hat = sigmoid(z)
    gradient = np.dot(X.T, (y_hat - y)) / m
    theta -= learning_rate * gradient

    loss = binary_cross_entropy(y, y_hat)
    losses.append(loss)

  return theta, losses

In [ ]:
# sigmoid in NumPy
def sigmoid(z):
  return 1 / (1 + np.exp(-z))

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score


n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

model = RandomForestClassifier(
      n_estimators=320,  # Number of trees
      max_depth=4
  )

lr_losses = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train
    theta, losses = gradient_descent(X_train, y_train, learning_rate=0.5, n_iters=500)

    # # Validate
    # y_pred_proba = sigmoid(np.dot(X_test, theta))
    # y_pred = (y_pred_proba >= 0.5).astype(int)
    # # y_pred = model.predict(X_test_scaled)
    # mae = mean_absolute_error(y_test, y_pred)
    # Validate
    y_pred = np.dot(X_test.values, theta)

    # Calculate evaluation metrics
    mse = sklearn_mse(y_test, y_pred)

    # Store results
    lr_losses.append(losses)

# Calculate average loss across folds
avg_loss = np.mean(lr_losses, axis=0)

plt.figure(figsize=(10, 6))
plt.plot(avg_loss, label='Logistic Regression Loss', color='purple')
plt.title('Average Logistic Regression Loss Curve (Across 5 Folds)')
plt.xlabel('Iteration')
plt.ylabel('Binary Cross-Entropy Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': X,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
data.Delivery_Time.hist()

In [ ]:
# Task Bonus: Write your code here: